In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

# Dataset
texts = [
    "I love this movie","This is wonderful","Absolutely fantastic film","I really enjoyed this movie",
    "Brilliant acting and story","Amazing experience","Superb direction","I liked this a lot",
    "Great movie overall","One of the best films",
    "I hate this movie","This is terrible","Absolutely boring film","I really disliked this movie",
    "Awful acting and story","Horrible experience","Poor direction","I did not like this at all",
    "Worst movie ever","Very disappointing film"
]

labels = [1]*10 + [0]*10

# Load model
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Prediction function
def predict(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    return logits.softmax(dim=-1)

# Before Fine-Tuning
print("Before Fine-Tuning:")
print("Positive:", predict("I love this movie"))
print("Negative:", predict("This is terrible"))

# Apply LoRA
config = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, target_modules=["q_lin","v_lin"])
model = get_peft_model(model, config)

# Training
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
model.train()

for epoch in range(5):
    for text, label in zip(texts, labels):
        inputs = tokenizer(text, return_tensors="pt")
        loss = model(**inputs, labels=torch.tensor([label])).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# After Fine-Tuning
model.eval()

print("\nAfter Fine-Tuning:")
print("Positive:", predict("I love this movie"))
print("Negative:", predict("This is terrible"))

W0410 06:47:49.587999 11800 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


ImportError: cannot import name 'clear_device_cache' from 'accelerate.utils.memory' (c:\Users\vishn\AppData\Local\Programs\Python\Python310\lib\site-packages\accelerate\utils\memory.py)